In [ ]:
import asyncio
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, learning_curve
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
from clients import Clients
from mongo.schemas import PoolsSnapshotTrainRegressionV2, PoolsSnapshotTestRegressionV2

# Kết nối MongoDB
mongo_client = Clients.get_mongo_client()

In [ ]:
async def load_data_v2():
    await mongo_client.initialize()
    
    print("Loading Train V2 data...")
    train_docs = await PoolsSnapshotTrainRegressionV2.find_many().to_list()
    df_train = pd.DataFrame([doc.model_dump() for doc in train_docs])
    
    print("Loading Test V2 data...")
    test_docs = await PoolsSnapshotTestRegressionV2.find_many().to_list()
    df_test = pd.DataFrame([doc.model_dump() for doc in test_docs])
    
    return df_train, df_test

In [ ]:
# Load data 
df_train, df_test = await load_data_v2()

print(f"Train shape: {df_train.shape}")
print(f"Test shape: {df_test.shape}")

# Kiểm tra qua dữ liệu
df_train.head()

In [ ]:
# ## 3. Feature Engineering (Define Features & Target)
# Danh sách Feature mới (8 features)
FEATURE_COLUMNS = [
    'log_tvl',          # Quy mô vốn (Logarit)
    'tvl_change_7d',    # Xu hướng dòng tiền
    'max_drawdown',     # Mức sụt giảm sâu nhất
    'tvl_volatility',   # Độ biến động TVL
    'apy_mean',         # Lợi nhuận trung bình
    'apy_std',          # Độ biến động lợi nhuận
    'chain_score',      # Điểm rủi ro Chain
    'token_score'       # Điểm uy tín Token (Quan trọng)
]

TARGET_COLUMN = 'risk_score'


# Tách X, y
X_train = df_train[FEATURE_COLUMNS]
y_train = df_train[TARGET_COLUMN]

X_test = df_test[FEATURE_COLUMNS]
y_test = df_test[TARGET_COLUMN]

print(f"Features sử dụng ({len(FEATURE_COLUMNS)}): {FEATURE_COLUMNS}")

In [ ]:
# ## 4. Baseline Model (Random Forest)

# Khởi tạo model với cấu hình cơ bản
rf_v2 = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

# Train
print("Training Baseline Model...")
rf_v2.fit(X_train, y_train)

# Predict
y_pred = rf_v2.predict(X_test)

# Đánh giá
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("-" * 30)
print(f"BASELINE METRICS (V2 Data):")
print(f"MAE: {mae:.4f} (Sai số trung bình)")
print(f"RMSE: {rmse:.4f}")
print(f"R2 Score: {r2:.4f}")
print("-" * 30)

In [ ]:
### 5. Phân tích Feature Importance

importances = rf_v2.feature_importances_
indices = np.argsort(importances)[::-1]

# Vẽ biểu đồ
plt.figure(figsize=(10, 6))
plt.title("Feature Importance (V2 Model)")
plt.bar(range(X_train.shape[1]), importances[indices], align="center", color='skyblue')
plt.xticks(range(X_train.shape[1]), [FEATURE_COLUMNS[i] for i in indices], rotation=45)
plt.tight_layout()
plt.show()

print("Độ quan trọng của các đặc trưng:")
for i in range(X_train.shape[1]):
    print(f"{i+1}. {FEATURE_COLUMNS[indices[i]]}: {importances[indices[i]]:.4f}")

In [ ]:
# ## 6. Hyperparameter Tuning (Grid Search)
# Tinh chỉnh để tìm bộ tham số tốt nhất.

param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [10, 20, None],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', None]
}

print("Đang chạy Grid Search (có thể mất vài phút)...")
grid_search = GridSearchCV(
    estimator=RandomForestRegressor(random_state=42),
    param_grid=param_grid,
    cv=3, # Cross validation 3-fold để nhanh hơn
    n_jobs=-1,
    scoring='neg_mean_absolute_error',
    verbose=1
)

grid_search.fit(X_train, y_train)

best_model_v2 = grid_search.best_estimator_
print(f"\nBest Params: {grid_search.best_params_}")

# Đánh giá lại với model tốt nhất
y_pred_opt = best_model_v2.predict(X_test)
mae_opt = mean_absolute_error(y_test, y_pred_opt)
r2_opt = r2_score(y_test, y_pred_opt)

print(f"Optimized MAE: {mae_opt:.4f}")
print(f"Optimized R2: {r2_opt:.4f}")

In [ ]:
# ## 7. Learning Curve (Kiểm tra Overfitting)
# Xem khoảng cách giữa Train score và CV score có lớn không.

train_sizes, train_scores, test_scores = learning_curve(
    best_model_v2, X_train, y_train, 
    cv=3, scoring='neg_mean_squared_error',
    n_jobs=-1, train_sizes=np.linspace(0.1, 1.0, 5)
)

train_rmse = np.sqrt(-np.mean(train_scores, axis=1))
test_rmse = np.sqrt(-np.mean(test_scores, axis=1))

plt.figure(figsize=(8, 5))
plt.title("Learning Curve (V2)")
plt.xlabel("Training examples")
plt.ylabel("RMSE")
plt.grid()
plt.plot(train_sizes, train_rmse, 'o-', color="r", label="Train RMSE")
plt.plot(train_sizes, test_rmse, 'o-', color="g", label="Validation RMSE")
plt.legend(loc="best")
plt.show()

In [ ]:
# ## 8. Lưu Model và Test thử
# Lưu lại model để dùng cho API.

# Lưu model
model_filename = 'risk_score_model_v2.pkl'
joblib.dump(best_model_v2, model_filename)
print(f"Đã lưu model vào: {model_filename}")

# Test thử một vài case điển hình từ tập Test
print("\n--- Kiểm tra thực tế một số mẫu ---")
sample_indices = np.random.choice(len(X_test), 5, replace=False)
for i in sample_indices:
    sample_X = X_test.iloc[i]
    true_y = y_test.iloc[i]
    pred_y = best_model_v2.predict([sample_X])[0]
    
    print(f"Pool Index {i}:")
    print(f" - Input: TVL={10**sample_X['log_tvl']:.0f}, Drawdown={sample_X['max_drawdown']:.2%}, TokenScore={sample_X['token_score']}")
    print(f" - Thực tế: {true_y:.2f} | Dự đoán: {pred_y:.2f} | Sai lệch: {abs(true_y - pred_y):.2f}")
    print("-" * 20)